<a href="https://colab.research.google.com/github/wiz124/chem169-git/blob/main/Li_Harry_RID_018_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#Exercise 0
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def count_amino_acids(sequence):
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    sequence=str(sequence)
    return {aa: sequence.count(aa) for aa in amino_acids}

df=pd.read_excel('mtb_with_localization.xlsx')

location=[]
for entry in df['Subcellular location [CC]']:
  if 'membrane' in str(entry).lower():
    location.append('membrane')
  elif 'secreted' in str(entry).lower():
    location.append('secreted')
  elif 'cytoplasm' in str(entry).lower():
    location.append(  'cytoplasm')
  else:
    location.append(None)

df['location']=location
df.dropna(subset=['location'], inplace=True)

y = df[['location']].to_numpy()

amino_acids = list('ACDEFGHIKLMNPQRSTVWY')

aa_counts = pd.DataFrame(
    df['Sequence'].apply(count_amino_acids).tolist(),
    columns=amino_acids,
    index=df.index
)
df = pd.concat([df, aa_counts], axis=1)


x = df[['Length'] + amino_acids].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,      # 30% for testing
    random_state=42,    # for reproducibility
    stratify=y          # maintain class distribution
)


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [10]:
#Exercise 1
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder


logmodel = LogisticRegression(solver='liblinear',max_iter=1000)
logmodel.fit(X_train, y_train)

forestmodel = RandomForestClassifier(n_estimators=100, random_state=42)
forestmodel.fit(X_train,y_train)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgbmodel = xgb.XGBClassifier(n_estimators=100,
                             random_state=42,
                             )
xgbmodel.fit(X_train, y_train_enc)




/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), 

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [12]:
#Exercise 2

log_pred = logmodel.predict(X_test)
forest_pred = forestmodel.predict(X_test)
xgb_pred = xgbmodel.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)
forest_accuracy = accuracy_score(y_test, forest_pred)

xgb_accuracy = accuracy_score(y_test_enc,xgb_pred)

results={
    'Logistic Regression':log_accuracy,
    'Random Forest':forest_accuracy,
    'XGBoost':xgb_accuracy
}
resultdf=pd.DataFrame(results,index=['Accuracy'])
display(resultdf)

,Logistic Regression,Random Forest,XGBoost
Accuracy,0.776256,0.712329,0.739726


Exercise 2

Logistic regression had the highest accuracy while random forest had the lowest. The differences are small and logistic regression being the biggest did not surprise me.

Exercise 3

Q1: logistic regression would be expected to do well.

Q2: Xgboost would do better

Q3: A simpler model would be more interpretable and efficient.

Exercise 5

Q1: Logistic regression won with this dataset because the data correlates closely to a linear relationship. Thus, the other models decision making is lower due to the overcomplexity of the solution.

Q2: I would choose logistic regression when the data shows a strong linear relationship.

Q3: Other features that might help improve predictions would be hydrophobicity or count of residue types.